# **Tutorial 09 - Neural Networks as Function Approximators: Width vs Depth**: Supplementary Interactive Material

-------------------------------------------------------------------------

So far every fit has started from a model with a known form. A neural network
is a flexible family of functions for when we only have samples, fitted the
same way as before: minimize a loss with gradient-based updates.

1. Build a curve by hand from neurons (Section 1).
2. Build a spike in two dimensions, and copy it by composing layers (Section 2).
3. Train a **wide** and a **deep** network with the same number of parameters
   on a two-dimensional distribution, and compare them (Sections 3–5).

Run the cells in order; training takes well under a minute.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.special import expit   # the sigmoid 1 / (1 + e^{-z}), without overflow warnings
import ipywidgets as widgets
from IPython.display import HTML, display

## 1. Building a curve from neurons

A **neuron** passes a linear function of its input through a nonlinear
**activation**. With the sigmoid $\sigma(z)=1/(1+e^{-z})$, the neuron
$\sigma\big(w(x-c)\big)$ is a soft step at $x=c$ whose steepness is set by $w$.
A network with one hidden layer is a weighted sum of $N$ such steps:

$$ \hat f(x)=a_0+\sum_{k=1}^{N} a_k\,\sigma\big(w(x-c_k)\big). $$

We can pick the weights by hand. To approximate
$f(x) = \log x + \sin^3 x + \tfrac{1}{3}\cos^3(x-2)$ on $[1,6]$, space the
steps evenly and give step $k$ the height $a_k$ by which $f$ changes across
its interval. The lower panel shows each weighted neuron, the upper panel
their sum.

**Increase the number of neurons. Does the error keep shrinking? What does
lowering the steepness do?**

In [ ]:
def f(x):
    return np.log(x) + np.sin(x)**3 + (1/3) * np.cos(x - 2)**3

x = np.linspace(1, 6, 400)


def sigmoid_staircase(x, n_neurons, steepness):
    nodes = np.linspace(1, 6, n_neurons + 1)
    centres = (nodes[:-1] + nodes[1:]) / 2               # c_k: where each step sits
    heights = np.diff(f(nodes))                          # a_k = f(t_k) - f(t_{k-1})
    w = steepness / (nodes[1] - nodes[0])                # input weight
    neurons = heights[:, None] * expit(w * (x[None, :] - centres[:, None]))  # a_k sigma(w(x - c_k))
    return f(nodes[0]) + neurons.sum(axis=0), neurons


def plot_staircase(n_neurons=5, steepness=10):
    approx, neurons = sigmoid_staircase(x, n_neurons, steepness)
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                                      gridspec_kw={'height_ratios': [3, 2]})
    top.plot(x, f(x), 'b-', linewidth=2, alpha=0.7, label='f(x)')
    top.plot(x, approx, 'r--', linewidth=2, label=f'Sum of {n_neurons} neurons')
    top.set(ylabel='y', ylim=(0, 3),
            title=f'max |error| = {np.max(np.abs(approx - f(x))):.3f}')
    top.legend(loc='lower right')
    bottom.plot(x, neurons.T, linewidth=1.2)
    bottom.axhline(0, color='k', linewidth=0.8)
    bottom.set(xlabel='x', ylabel=r'$a_k\,\sigma(w(x-c_k))$', xlim=(1, 6))
    for ax in (top, bottom):
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


neuron_slider = widgets.IntSlider(value=5, min=1, max=40, description='Neurons',
                                  continuous_update=False)
steepness_slider = widgets.FloatSlider(value=10, min=0.5, max=40, step=0.5, description='Steepness',
                                       continuous_update=False)
output = widgets.interactive_output(plot_staircase, {'n_neurons': neuron_slider,
                                                     'steepness': steepness_slider})
display(widgets.VBox([widgets.HBox([neuron_slider, steepness_slider]), output]))

More steps follow $f$ more closely, and this works for any continuous $f$.
That is the idea behind the **universal approximation theorem**: one hidden
layer with enough neurons can approximate any continuous function on a bounded
domain. It does not say how many neurons are needed, or whether training will
find the weights.

## 2. Two dimensions: a spike, then copies of it

From here on we use the **ReLU** activation, $\mathrm{relu}(z)=\max(0,z)$, the
usual choice for training. With two inputs, a neuron's linear function
$w_1x+w_2y+b$ is a **plane**, and the ReLU flattens the part below zero into
a ramp (panel 1). Four such neurons give

$$ \mathrm{relu}(x)+\mathrm{relu}(-x)+\mathrm{relu}(y)+\mathrm{relu}(-y)=|x|+|y|, $$

so $1-|x|-|y|$ is an upside-down funnel with four planar faces (panel 2). It
is not yet local: the faces run off to $-\infty$. One more ReLU cuts
everything below zero and leaves a **spike**
$s(x,y)=\mathrm{relu}(1-|x|-|y|)$ (panel 3). A weighted sum of spikes can
approximate a surface just as steps approximated a curve.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-2, 2, 161), np.linspace(-2, 2, 161))

def relu(z):
    return np.maximum(z, 0)

def absolute(t):
    return relu(t) + relu(-t)                    # |t| from two ReLU neurons

def spike(u, v):
    return relu(1 - absolute(u) - absolute(v))   # 4 neurons, then 1 more

def plot_surfaces(surfaces):
    fig = plt.figure(figsize=(4 * len(surfaces), 3.8))
    for i, (z, title) in enumerate(surfaces, start=1):
        ax = fig.add_subplot(1, len(surfaces), i, projection='3d')
        ax.plot_surface(xx, yy, z, cmap='viridis', rcount=161, ccount=161,
                        linewidth=0, antialiased=False)
        ax.set(xlabel='x', ylabel='y', title=title)
        ax.view_init(elev=35, azim=-60)
    plt.tight_layout()
    plt.show()

plot_surfaces([(relu(xx), '1. One neuron: relu$(x)$'),
               (1 - absolute(xx) - absolute(yy), '2. Four neurons: $1-|x|-|y|$'),
               (spike(xx, yy), '3. One more ReLU: a spike')])

The spike needed a second layer. Layers can do more than that: the same four
neurons also give the **fold** $F(x,y)=(2|x|-2,\;2|y|-2)$, which sends four
different points to the same place. Feeding $F$ into the spike therefore
repeats it four times, and folding twice repeats it sixteen times. Each fold
costs four neurons and quadruples the number of spikes.

In [ ]:
def fold(u, v):
    return 2 * absolute(u) - 2, 2 * absolute(v) - 2   # one layer of 4 neurons

plot_surfaces([(spike(*fold(xx, yy)), r'$s\circ F$: 9 neurons'),
               (spike(*fold(*fold(xx, yy))), r'$s\circ F\circ F$: 13 neurons')])

A single hidden layer cannot reuse anything: it must spend separate neurons on
every spike. Depth helps when the target has structure that can be built once
and reused.

> **Explore:** Add a third fold: how many spikes, from how many neurons?
> Then replace `spike(u, v)` by the off-centre `spike(u - 0.5, v)`. Why are
> neighbouring copies mirror images?

## 3. The target: a two-dimensional distribution

We now train networks to fit the (unnormalized) density

$$ p(x,y)=e^{-(r-2)^2/(2\cdot 0.25^2)}\,\big(0.6+0.4\cos3\theta\big)+e^{-r^2/(2\cdot 0.35^2)},
\qquad (x,y)\in[-3,3]^2, $$

a ring with three lobes around a narrow central peak. The networks only see
its value at 1000 random points. We judge each fit on a $101\times101$ grid of
points that are never used for training.

In [ ]:
def p(x, y):
    r, theta = np.hypot(x, y), np.arctan2(y, x)
    ring = np.exp(-(r - 2)**2 / (2 * 0.25**2)) * (0.6 + 0.4 * np.cos(3 * theta))
    peak = np.exp(-r**2 / (2 * 0.35**2))
    return ring + peak

rng = np.random.default_rng(0)
samples = rng.uniform(-3, 3, size=(1000, 2))      # training inputs, one (x, y) per row
values = p(samples[:, 0], samples[:, 1])          # training targets

grid = np.linspace(-3, 3, 101)
grid_x, grid_y = np.meshgrid(grid, grid)          # evaluation grid
grid_p = p(grid_x, grid_y)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)
image = axes[0].imshow(grid_p, extent=(-3, 3, -3, 3), origin='lower', cmap='viridis', vmin=0, vmax=1)
axes[0].set(xlabel='x', ylabel='y', title='Target p(x, y)')
axes[1].scatter(samples[:, 0], samples[:, 1], c=values, s=4, cmap='viridis', vmin=0, vmax=1)
axes[1].set(xlabel='x', title='The 1000 training samples', xlim=(-3, 3), ylim=(-3, 3), aspect='equal')
fig.colorbar(image, ax=axes, shrink=0.85)
plt.show()

## 4. Data, model, and training loop in PyTorch

**Data.** Tensors hold one sample per row. A `DataLoader` serves shuffled
**minibatches** of 64; one pass through all of them is an **epoch**.

In [ ]:
X_train = torch.tensor(samples, dtype=torch.float32)                # shape (1000, 2)
y_train = torch.tensor(values, dtype=torch.float32).unsqueeze(1)    # shape (1000, 1)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

X_grid = torch.tensor(np.column_stack([grid_x.ravel(), grid_y.ravel()]), dtype=torch.float32)
y_grid = torch.tensor(grid_p.ravel(), dtype=torch.float32).unsqueeze(1)

print(f"Training inputs: {tuple(X_train.shape)}, targets: {tuple(y_train.shape)}, "
      f"minibatches per epoch: {len(train_loader)}")

**Model.** Each `nn.Linear` is an affine map $Wz+b$, followed by a ReLU on
every layer but the last. `hidden_sizes` sets the architecture: `[335]` is one
hidden layer of 335 neurons, `[20, 20, 20, 20]` is four layers of 20.

In [ ]:
class FunctionApproximator(nn.Module):

    def __init__(self, hidden_sizes):
        super().__init__()
        sizes = [2] + list(hidden_sizes) + [1]   # 2 inputs -> hidden layers -> 1 output
        self.layers = nn.ModuleList(
            nn.Linear(n_in, n_out) for n_in, n_out in zip(sizes[:-1], sizes[1:])
        )

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = torch.relu(layer(x))   # affine map, then ReLU
        return self.layers[-1](x)      # no activation on the output


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

**Training loop.** We minimize the mean squared error with **Adam**, a variant
of gradient descent with a step size per parameter. Each update clears the old
gradients, evaluates the loss, backpropagates, and steps. After each epoch we
record the fit on the evaluation grid.

In [ ]:
def train_model(model, train_loader, epochs=100, learning_rate=0.004):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    prediction_history, loss_history = [], []
    for epoch in range(epochs):
        for data, target in train_loader:        # one pass through the shuffled minibatches
            optimizer.zero_grad()
            loss = criterion(model(data), target)
            loss.backward()
            optimizer.step()

        with torch.no_grad():                    # record the fit on the evaluation grid
            predictions = model(X_grid)
            loss_history.append(criterion(predictions, y_grid).item())
            prediction_history.append(predictions.numpy().reshape(grid_p.shape))

    return np.array(prediction_history), np.array(loss_history)

## 5. Width vs depth

Both networks get the same **1341 parameters**, the same samples, and the same
optimizer settings:

- **Wide:** one hidden layer of 335 neurons.
- **Deep:** four hidden layers of 20 neurons, 80 in total.

**Before running the cell: which do you expect to fit $p$ better, the network
with four times as many neurons or the one that composes them in layers?**

In [ ]:
architectures = {
    "Wide: 1 x 335": [335],
    "Deep: 4 x 20": [20, 20, 20, 20],
}

results = {}
for name, hidden_sizes in architectures.items():
    torch.manual_seed(0)                     # same initialization and batch order on each rerun
    model = FunctionApproximator(hidden_sizes)
    predictions, losses = train_model(model, train_loader, epochs=100, learning_rate=0.004)
    results[name] = {"predictions": predictions, "losses": losses}
    print(f"{name:<15} hidden neurons = {sum(hidden_sizes):4d}, "
          f"parameters = {count_parameters(model):6d}, final grid MSE = {losses[-1]:.2e}")

The animation shows each prediction on the evaluation grid after every epoch.

**Which features appear first? Which network resolves the three lobes?**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True)
style = dict(extent=(-3, 3, -3, 3), origin='lower', cmap='viridis', vmin=0, vmax=1)
axes[0].imshow(grid_p, **style)
axes[0].set(xlabel='x', ylabel='y', title='Target')
images, labels = [], []
for ax, name in zip(axes[1:], results):
    images.append(ax.imshow(np.zeros_like(grid_p), **style))
    labels.append(ax.text(0.03, 0.93, '', transform=ax.transAxes, fontsize=10,
                          bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8)))
    ax.set(xlabel='x', title=name)
plt.tight_layout()

def update(frame):
    for image, label, result in zip(images, labels, results.values()):
        image.set_data(result["predictions"][frame])
        label.set_text(f'Epoch = {frame + 1}, MSE = {result["losses"][frame]:.1e}')
    return (*images, *labels)

n_frames = len(next(iter(results.values()))["predictions"])
ani = FuncAnimation(fig, update, frames=n_frames, interval=100, blit=True, repeat=True)
plt.close(fig)
HTML(ani.to_jshtml())

The grid MSE after every epoch, and the error $\hat p-p$ after the last one:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
errors = {name: result["predictions"][-1] - grid_p for name, result in results.items()}
limit = max(np.abs(error).max() for error in errors.values())   # shared colour scale

for name, result in results.items():
    axes[0].semilogy(np.arange(1, len(result["losses"]) + 1), result["losses"], label=name)
axes[0].set(xlabel='Epoch', ylabel='Grid MSE', title='Loss on the evaluation grid')
axes[0].grid(alpha=0.3)
axes[0].legend()

for ax, (name, error) in zip(axes[1:], errors.items()):
    image = ax.imshow(error, extent=(-3, 3, -3, 3), origin='lower', cmap='RdBu_r',
                      vmin=-limit, vmax=limit)
    ax.set(xlabel='x', ylabel='y', title=f'{name}: prediction - target')
fig.colorbar(image, ax=axes[1:], shrink=0.85)
plt.show()

for name, error in errors.items():
    print(f"{name:<15} final grid MSE = {np.mean(error**2):.2e}, max |error| = {np.max(np.abs(error)):.3f}")

### What does the comparison tell us?

With the same parameter budget, the deep network wins clearly on this run: its
grid MSE ends near $10^{-3}$, about 15 times below the wide network's
$1.5\times10^{-2}$. The wide network's fit is a blurred version of the target:
its central peak reaches only about $0.6$ and its lobes about $0.7$, instead
of $1$.

Section 2 suggests why. Each neuron of the wide network is a ramp that varies
in one direction across the whole plane, so a ring or a peak has to come from
many ramps cancelling. In the deep network, later layers build on features
that earlier layers have already made. A single hidden layer *can* represent
$p$ given enough neurons; at this budget, depth used the parameters better.

> **Explore:** Change `torch.manual_seed(0)` to `1` and `2`. Does the same
> network come out ahead?
>
> How wide must the single hidden layer be to match the deep network? Try
> `[1000]` and `[3000]`, and compare the parameter counts.
>
> Change the 1000 training samples in Section 3 to 250, then to 4000, and
> rerun from there. What changes?

> **Discuss:** Which parts of the gradient descent workflow from Tutorials 03
> and 07 carry over unchanged, and what has PyTorch automated for us?